# Create experiment info

Writes `SAMPLE_DIR/metadata/experiment_info.yaml`, a per-experiment record
matching the master experiment-info CSVs (e.g. `lt_experiment_info.csv`), so
it can be collected back into them (`collect_experiment_info`).

Run after `before_imaging` 01 (bits HAL config) and 03
(`round_bit_color_map.csv`), and before `create_merlin_files.ipynb`.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR     = Path(os.getcwd()).parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/after_imaging/create_submit_files/)
SAMPLE_DIR    = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_26/
METADATA_DIR  = SAMPLE_DIR / "metadata"
SETTINGS_DIR  = SAMPLE_DIR / "settings"
POSITIONS_DIR = SAMPLE_DIR / "positions"
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import ExperimentInfo, save_experiment_info, resolve_sample_identity, resolve_data_home
from MERci.acquisition.configs    import read_hal_exposure_time, get_acquisition_type
from MERci.acquisition.pipeline_config import load_pipeline_config

PIPELINE_ID     = "tumor_epi"   # EDIT ME -- one of the ids in pipeline_export.PIPELINES
PIPELINE_CONFIG = load_pipeline_config(MERCI_DIR / "data" / "pipelines" / f"{PIPELINE_ID}_pipeline.yaml")

MICROSCOPE = PIPELINE_CONFIG.microscope   # must match before_imaging notebooks 01/03
OBJECTIVE  = PIPELINE_CONFIG.objective    # must match before_imaging notebook 02

# SAMPLE_NAME/IMAGING_DIR name the files the before_imaging notebooks wrote.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)

# Acquisition type ("epi" or "disk") is derived from MICROSCOPE, not typed by hand.
ACQUISITION_TYPE = get_acquisition_type(MICROSCOPE)

print(f"SAMPLE_DIR       : {SAMPLE_DIR}")
print(f"SAMPLE_NAME      : {SAMPLE_NAME}")
print(f"IMAGING_DIR      : {IMAGING_DIR!r}")
print(f"ACQUISITION_TYPE : {ACQUISITION_TYPE}")

## Auto-filled fields

Read from what `before_imaging` 01 and 03 wrote.

In [ ]:
# Bit count from round_bit_color_map.csv (before_imaging 03).
rbc_path = METADATA_DIR / "round_bit_color_map.csv"
n_bits = None
if rbc_path.exists():
    rbc_df = pd.read_csv(rbc_path)
    n_bits = int(rbc_df["round"].max())
else:
    print(f"WARNING: {rbc_path} not found -- run before_imaging notebook 03 first. n_bits left as None.")

# Exposure from the bits HAL config (before_imaging 01).
bits_hal_configs = sorted(SETTINGS_DIR.glob("hal-config-*bits*.xml"))
exposure_s = read_hal_exposure_time(bits_hal_configs[-1]) if bits_hal_configs else None
if exposure_s is None:
    print("WARNING: could not read exposure_time from a bits HAL config in settings/.")

# Positions file(s) actually present for this sample (single- or per-tissue layout).
positions_files = sorted(POSITIONS_DIR.glob(f"positions_{SAMPLE_NAME}*.txt"))

print(f"n_bits (hyb rounds) : {n_bits}")
print(f"exposure (ms)       : {exposure_s * 1000 if exposure_s else None}")
print(f"positions files     : {[p.name for p in positions_files]}")

## Fields MERci can't know

Cluster paths and sample metadata: fill these in.

In [ ]:
# ── Fields MERci can't know ───────────────────────────────────────────────
# Merlin pipelines take PROJECT/LIB_NAME/N_OPT from pipeline.yaml; fishtank
# pipelines set them here.
if PIPELINE_CONFIG.merlin:
    PROJECT  = PIPELINE_CONFIG.merlin.project
    LIB_NAME = PIPELINE_CONFIG.merlin.lib_name
    N_OPT    = PIPELINE_CONFIG.merlin.n_optimize_iterations
else:
    PROJECT  = "lt"    # EDIT ME -- "bc" | "lt" | "mf": which master CSV this matches
    LIB_NAME = "LT2"   # EDIT ME -- codebook/probe library
    N_OPT    = 10      # EDIT ME -- unused by fishtank, kept for the shared schema

# Override IMAGING_DIR only if it was detected wrong:
# IMAGING_DIR = "epi"

# Cluster paths. DATA_HOME comes from where this clone lives (resolve_data_home);
# the others match the master CSV's merlin_home/folder_name columns.
DATA_HOME   = resolve_data_home(MERCI_DIR)
MERLIN_HOME = (f"{DATA_HOME}/{SAMPLE_NAME}/{IMAGING_DIR}/merlin" if IMAGING_DIR
               else f"{DATA_HOME}/{SAMPLE_NAME}/merlin")
FOLDER_NAME = (f"{SAMPLE_NAME}/{IMAGING_DIR}/data" if IMAGING_DIR
               else f"{SAMPLE_NAME}/data")

# Sample metadata recorded in the master CSV.
EXTRA = {
    "hyb_temp":         "",   # e.g. "37C, 2d"
    "fix_type":         "",   # e.g. "4%PFA, 15 min"
    "permeabilization": "",   # e.g. "70%EtOH ON"
    "sample_type":      "",   # e.g. embryo stage
    "source":           "",   # institution/collaborator
    "thickness":        "",   # tissue thickness in microns
    "exposure":         exposure_s * 1000 if exposure_s else None,   # ms, auto-filled above
    "n_opt":            N_OPT,
    "imaging_dir":      IMAGING_DIR,
    "objective":        OBJECTIVE,
    "positions_name":   [p.name for p in positions_files],
}

print("PROJECT    :", PROJECT)
print("LIB_NAME   :", LIB_NAME)
print("N_OPT      :", N_OPT)
print("IMAGING_DIR:", repr(IMAGING_DIR))
print("DATA_HOME  :", DATA_HOME)
print("MERLIN_HOME:", MERLIN_HOME)
print("FOLDER_NAME:", FOLDER_NAME)
print("EXTRA      :", EXTRA)

## Save

In [ ]:
info = ExperimentInfo(
    sample_name      = SAMPLE_NAME,
    project          = PROJECT,
    microscope       = MICROSCOPE,
    acquisition_type = ACQUISITION_TYPE,
    lib_name         = LIB_NAME,
    data_home        = DATA_HOME,
    merlin_home      = MERLIN_HOME,
    folder_name      = FOLDER_NAME,
    extra            = EXTRA,
)

out_path = METADATA_DIR / "experiment_info.yaml"
save_experiment_info(info, out_path)
print(f"Saved: {out_path}\n")
print(out_path.read_text())